# Análisis Exploratorio de Datos (EDA) — SPREE
**Sistema Predictivo de Retención Estudiantil**

Este notebook analiza el dataset sintético de 500 estudiantes para comprender las variables asociadas a la deserción estudiantil. El objetivo es identificar patrones, distribuciones, correlaciones y el grado de desbalanceo de clases antes del modelado ML.

---
**Tarea:** SPR-10 — T1.3 Análisis exploratorio (EDA)  
**Sprint:** SCRUM Sprint 1  
**Criterios de aceptación:** Notebook con al menos 8 gráficas, comentarios de hallazgos clave

## 0. Configuración e imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Estilo visual
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f9fa'
plt.rcParams['font.family'] = 'DejaVu Sans'
PALETTE = {'Activo': '#2196F3', 'Desertor': '#F44336'}

print('Librerías cargadas correctamente.')

## 1. Carga y unión de datos

In [ ]:
import os

# Ruta absoluta al directorio data/raw (compatible con ejecución desde cualquier directorio)
NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd()
BASE = os.path.join(NOTEBOOK_DIR, "..", "..", "data", "raw")
# Fallback: ruta absoluta directa
if not os.path.exists(os.path.join(BASE, "demographic.csv")):
    BASE = os.path.join(os.getcwd(), "data", "raw")

demographic   = pd.read_csv(os.path.join(BASE, "demographic.csv"))
academics     = pd.read_csv(os.path.join(BASE, "academics.csv"))
financial     = pd.read_csv(os.path.join(BASE, "financial.csv"))
socioeconomic = pd.read_csv(os.path.join(BASE, "socioeconomic.csv"))
well_being    = pd.read_csv(os.path.join(BASE, "well-being.csv"))
additional    = pd.read_csv(os.path.join(BASE, "additional.csv"))

df = demographic.merge(academics,     on="id_estudiante") \
                .merge(financial,     on="id_estudiante") \
                .merge(socioeconomic, on="id_estudiante") \
                .merge(well_being,    on="id_estudiante") \
                .merge(additional,    on="id_estudiante")

df = df.drop(columns=["historial_notas"], errors="ignore")

print(f"Dataset unificado: {df.shape[0]} estudiantes, {df.shape[1]} variables")
df.head()


## 2. Resumen estadístico

In [ ]:
print('=== Información general ===')
print(df.info())
print()
print('=== Valores nulos por columna ===')
print(df.isnull().sum())
print()
print('=== Estadísticas descriptivas (variables numéricas) ===')
df.describe().round(2)

---
## Gráfica 1 — Distribución de la variable objetivo (desbalanceo de clases)

**Objetivo:** Verificar si el dataset está balanceado entre Activos y Desertores.

**¿Por qué importa?** Si hay muchos más Activos que Desertores (o viceversa), el modelo de ML puede aprender a predecir siempre la clase mayoritaria y tener alta precisión sin haber aprendido nada útil. Esto se llama *desbalanceo de clases* y hay que detectarlo y corregirlo antes de modelar.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

counts = df['estado_estudiante'].value_counts()
colors = [PALETTE[c] for c in counts.index]

# Barras
bars = axes[0].bar(counts.index, counts.values, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f'{val}\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].set_title('Cantidad de estudiantes por estado', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Número de estudiantes')
axes[0].set_ylim(0, 420)

# Pie chart
axes[1].pie(counts.values, labels=counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proporción de clases', fontsize=13, fontweight='bold')

plt.suptitle('Gráfica 1 — Distribución de la variable objetivo', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('grafica_01_distribucion_objetivo.png', dpi=150, bbox_inches='tight')
plt.show()

ratio = counts['Activo'] / counts['Desertor']
print(f'\n📊 HALLAZGO: Ratio Activo/Desertor = {ratio:.2f}:1')
print(f'El dataset presenta desbalanceo moderado (70% activos vs 30% desertores).')
print(f'Se recomienda aplicar técnicas como SMOTE o class_weight en el modelo.')

---
## Gráfica 2 — Distribución del promedio académico por estado

**Objetivo:** Ver si el promedio académico difiere entre estudiantes activos y desertores.

**¿Por qué importa?** Si los desertores tienen consistentemente promedios más bajos, esta variable será muy útil para el modelo predictivo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for estado, color in PALETTE.items():
    subset = df[df['estado_estudiante'] == estado]['promedio_academico']
    axes[0].hist(subset, bins=20, alpha=0.7, color=color, label=estado, edgecolor='white')
axes[0].set_title('Histograma de promedio académico', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Promedio académico')
axes[0].set_ylabel('Frecuencia')
axes[0].legend()
axes[0].axvline(3.0, color='gray', linestyle='--', alpha=0.7, label='Mínimo aprobatorio')

# Boxplot
df.boxplot(column='promedio_academico', by='estado_estudiante', ax=axes[1],
           boxprops=dict(color='navy'),
           medianprops=dict(color='red', linewidth=2))
axes[1].set_title('Boxplot de promedio por estado', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Estado del estudiante')
axes[1].set_ylabel('Promedio académico')
plt.suptitle('')

plt.suptitle('Gráfica 2 — Promedio académico vs estado del estudiante', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('grafica_02_promedio_academico.png', dpi=150, bbox_inches='tight')
plt.show()

medias = df.groupby('estado_estudiante')['promedio_academico'].mean()
print('\n📊 HALLAZGO: Promedios académicos medios:')
for estado, media in medias.items():
    print(f'  {estado}: {media:.2f}')
print(f'  Diferencia: {abs(medias["Activo"] - medias["Desertor"]):.2f} puntos')
print('→ El promedio académico es un fuerte predictor de deserción.')

---
## Gráfica 3 — Asistencia a clases vs estado del estudiante

**Objetivo:** Comparar el porcentaje de asistencia entre activos y desertores.

**¿Por qué importa?** La baja asistencia es uno de los indicadores más tempranos de riesgo de deserción.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Violin plot
activos   = df[df['estado_estudiante'] == 'Activo']['asistencia_clases']
desertores = df[df['estado_estudiante'] == 'Desertor']['asistencia_clases']

vp = axes[0].violinplot([activos, desertores], positions=[1, 2], showmedians=True)
for i, (pc, color) in enumerate(zip(vp['bodies'], ['#2196F3', '#F44336'])):
    pc.set_facecolor(color)
    pc.set_alpha(0.7)
vp['cmedians'].set_color('black')
axes[0].set_xticks([1, 2])
axes[0].set_xticklabels(['Activo', 'Desertor'])
axes[0].set_title('Distribución de asistencia (violin)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Asistencia (%)')

# Barras de franjas de asistencia
bins_labels = ['< 60%', '60-75%', '75-90%', '> 90%']
bins_edges = [0, 60, 75, 90, 101]
df['franja_asistencia'] = pd.cut(df['asistencia_clases'], bins=bins_edges, labels=bins_labels, right=False)
tabla = df.groupby(['franja_asistencia', 'estado_estudiante']).size().unstack(fill_value=0)
tabla.plot(kind='bar', ax=axes[1], color=[PALETTE[c] for c in tabla.columns], edgecolor='white')
axes[1].set_title('Franjas de asistencia por estado', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Franja de asistencia')
axes[1].set_ylabel('Número de estudiantes')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Estado')

plt.suptitle('Gráfica 3 — Asistencia a clases vs estado del estudiante', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('grafica_03_asistencia.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 HALLAZGO:')
print(f'  Asistencia media Activos: {activos.mean():.1f}%')
print(f'  Asistencia media Desertores: {desertores.mean():.1f}%')
print('→ Los desertores tienen significativamente menor asistencia.')
print('→ Umbral crítico: por debajo del 75% el riesgo aumenta notoriamente.')

---
## Gráfica 4 — Variables financieras y mora en matrícula

**Objetivo:** Analizar cómo el estado financiero (pagos, mora, becas) se relaciona con la deserción.

**¿Por qué importa?** Las dificultades económicas son una de las principales causas de deserción en Colombia.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Estado de pagos
tabla_pago = df.groupby(['estado_pagos', 'estado_estudiante']).size().unstack(fill_value=0)
tabla_pago_pct = tabla_pago.div(tabla_pago.sum(axis=1), axis=0) * 100
tabla_pago_pct.plot(kind='bar', ax=axes[0], color=[PALETTE[c] for c in tabla_pago_pct.columns],
                    edgecolor='white')
axes[0].set_title('Estado de pagos', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Porcentaje (%)')
axes[0].tick_params(axis='x', rotation=15)
axes[0].legend(title='Estado', fontsize=9)

# Mora en matrícula
tabla_mora = df.groupby(['mora_matricula', 'estado_estudiante']).size().unstack(fill_value=0)
tabla_mora_pct = tabla_mora.div(tabla_mora.sum(axis=1), axis=0) * 100
tabla_mora_pct.plot(kind='bar', ax=axes[1], color=[PALETTE[c] for c in tabla_mora_pct.columns],
                    edgecolor='white')
axes[1].set_title('Mora en matrícula', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Porcentaje (%)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Estado', fontsize=9)

# Becas y apoyos
tabla_beca = df.groupby(['becas_apoyos', 'estado_estudiante']).size().unstack(fill_value=0)
tabla_beca_pct = tabla_beca.div(tabla_beca.sum(axis=1), axis=0) * 100
tabla_beca_pct.plot(kind='bar', ax=axes[2], color=[PALETTE[c] for c in tabla_beca_pct.columns],
                    edgecolor='white')
axes[2].set_title('Becas y apoyos', fontsize=12, fontweight='bold')
axes[2].set_ylabel('Porcentaje (%)')
axes[2].tick_params(axis='x', rotation=0)
axes[2].legend(title='Estado', fontsize=9)

plt.suptitle('Gráfica 4 — Variables financieras vs deserción', fontsize=14)
plt.tight_layout()
plt.savefig('grafica_04_financiero.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 HALLAZGO:')
print('→ Los estudiantes en mora presentan tasas de deserción superiores.')
print('→ Los estudiantes con beca tienen menor probabilidad de desertar.')
print('→ El estado financiero debe incluirse como variable predictora en el modelo.')

---
## Gráfica 5 — Perfil socioeconómico (estrato e ingresos)

**Objetivo:** Explorar si el estrato socioeconómico y los ingresos familiares predicen la deserción.

**¿Por qué importa?** En Colombia el estrato es un indicador clave de acceso a recursos educativos.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tasa de deserción por estrato
tasa_estrato = df.groupby('estrato')['estado_estudiante'].apply(
    lambda x: (x == 'Desertor').mean() * 100
).reset_index()
tasa_estrato.columns = ['estrato', 'tasa_desercion']
tasa_estrato = tasa_estrato.sort_values('estrato')

bars = axes[0].bar(tasa_estrato['estrato'].astype(str), tasa_estrato['tasa_desercion'],
                   color='#E53935', edgecolor='white', linewidth=1.2)
axes[0].axhline(30, color='gray', linestyle='--', alpha=0.7, label='Media global (30%)')
for bar, val in zip(bars, tasa_estrato['tasa_desercion']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10)
axes[0].set_title('Tasa de deserción por estrato', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Estrato socioeconómico')
axes[0].set_ylabel('Tasa de deserción (%)')
axes[0].legend()
axes[0].set_ylim(0, 60)

# Ingresos familiares por estado
for estado, color in PALETTE.items():
    subset = df[df['estado_estudiante'] == estado]['ingresos_familiares']
    axes[1].hist(subset / 1_000_000, bins=20, alpha=0.7, color=color, label=estado, edgecolor='white')
axes[1].set_title('Distribución de ingresos familiares', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Ingresos familiares (millones COP)')
axes[1].set_ylabel('Frecuencia')
axes[1].legend()

plt.suptitle('Gráfica 5 — Perfil socioeconómico vs deserción', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('grafica_05_socioeconomico.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 HALLAZGO:')
print(f'  Estrato con mayor tasa de deserción: {tasa_estrato.loc[tasa_estrato["tasa_desercion"].idxmax(), "estrato"]}')
print(f'  Estrato con menor tasa de deserción: {tasa_estrato.loc[tasa_estrato["tasa_desercion"].idxmin(), "estrato"]}')
print('→ El estrato bajo se asocia con mayor riesgo de deserción.')
print('→ Los ingresos familiares también muestran diferencia entre grupos.')

---
## Gráfica 6 — Materias perdidas y créditos aprobados

**Objetivo:** Analizar el rendimiento académico cuantitativo (materias perdidas, créditos aprobados) entre ambos grupos.

**¿Por qué importa?** El fracaso académico acumulado es un predictor clásico de abandono.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Materias perdidas
max_perd = int(df['materias_perdidas'].max())
for estado, color in PALETTE.items():
    subset = df[df['estado_estudiante'] == estado]['materias_perdidas']
    axes[0].hist(subset, bins=range(0, max_perd+2), alpha=0.7,
                 color=color, label=estado, edgecolor='white', density=True)
axes[0].set_title('Distribución de materias perdidas', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Número de materias perdidas')
axes[0].set_ylabel('Densidad')
axes[0].legend()

# Créditos aprobados vs matriculados (scatter)
for estado, color in PALETTE.items():
    sub = df[df['estado_estudiante'] == estado]
    axes[1].scatter(sub['creditos_matriculados'], sub['creditos_aprobados'],
                    alpha=0.4, color=color, label=estado, s=30)
axes[1].plot([0, 25], [0, 25], 'k--', alpha=0.4, label='100% aprobación')
axes[1].set_title('Créditos matriculados vs aprobados', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Créditos matriculados')
axes[1].set_ylabel('Créditos aprobados')
axes[1].legend()

plt.suptitle('Gráfica 6 — Rendimiento académico cuantitativo', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('grafica_06_creditos_materias.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 HALLAZGO:')
print(f'  Materias perdidas media Activos: {df[df["estado_estudiante"]=="Activo"]["materias_perdidas"].mean():.2f}')
print(f'  Materias perdidas media Desertores: {df[df["estado_estudiante"]=="Desertor"]["materias_perdidas"].mean():.2f}')
print('→ Los desertores pierden más materias y aprueban menos créditos en proporción.')
print('→ Estudiantes que aprueban menos del 70% de créditos matriculados son de alto riesgo.')

---
## Gráfica 7 — Mapa de correlaciones

**Objetivo:** Visualizar la correlación entre todas las variables numéricas.

**¿Por qué importa?** Identifica qué variables están relacionadas entre sí (multicolinealidad) y cuáles se correlacionan más con la deserción. Ayuda a seleccionar features para el modelo.

In [ ]:
# Codificar variable objetivo para incluirla en la correlación
df_corr = df.copy()
df_corr['desercion'] = (df_corr['estado_estudiante'] == 'Desertor').astype(int)

# Seleccionar solo columnas numéricas
numericas = df_corr.select_dtypes(include=[np.number]).columns.tolist()
corr_matrix = df_corr[numericas].corr()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))  # triángulo superior vacío
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Gráfica 7 — Mapa de correlaciones entre variables numéricas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('grafica_07_correlaciones.png', dpi=150, bbox_inches='tight')
plt.show()

# Variables más correlacionadas con deserción
cor_desercion = corr_matrix['desercion'].drop('desercion').sort_values(key=abs, ascending=False)
print('\n📊 HALLAZGO — Variables más correlacionadas con deserción:')
for var, val in cor_desercion.head(8).items():
    direction = '↑ riesgo' if val > 0 else '↓ riesgo'
    print(f'  {var:35s} r={val:+.3f}  {direction}')

---
## Gráfica 8 — Bienestar estudiantil y semestre cursado

**Objetivo:** Analizar el momento de deserción (¿en qué semestre ocurre más?) y la relación con atenciones de bienestar.

**¿Por qué importa?** Saber en qué semestre hay mayor deserción permite intervenciones preventivas oportunas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tasa de deserción por semestre
tasa_semestre = df.groupby('semestre_cursado')['estado_estudiante'].apply(
    lambda x: (x == 'Desertor').mean() * 100
).reset_index()
tasa_semestre.columns = ['semestre', 'tasa_desercion']

axes[0].plot(tasa_semestre['semestre'], tasa_semestre['tasa_desercion'],
             'o-', color='#E53935', linewidth=2, markersize=8)
axes[0].fill_between(tasa_semestre['semestre'], tasa_semestre['tasa_desercion'],
                     alpha=0.15, color='#E53935')
axes[0].axhline(30, color='gray', linestyle='--', alpha=0.6, label='Media global')
axes[0].set_title('Tasa de deserción por semestre', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Semestre cursado')
axes[0].set_ylabel('Tasa de deserción (%)')
axes[0].set_xticks(tasa_semestre['semestre'])
axes[0].legend()

# Atenciones psicológicas vs estado
for estado, color in PALETTE.items():
    subset = df[df['estado_estudiante'] == estado]['atenciones_psicologicas']
    axes[1].hist(subset, bins=15, alpha=0.7, color=color, label=estado, edgecolor='white', density=True)
axes[1].set_title('Atenciones psicológicas por estado', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Número de atenciones psicológicas')
axes[1].set_ylabel('Densidad')
axes[1].legend()

plt.suptitle('Gráfica 8 — Bienestar estudiantil y momento de deserción', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('grafica_08_bienestar_semestre.png', dpi=150, bbox_inches='tight')
plt.show()

sem_max = tasa_semestre.loc[tasa_semestre['tasa_desercion'].idxmax()]
print('\n📊 HALLAZGO:')
print(f'  Semestre de mayor riesgo: Semestre {int(sem_max["semestre"])} ({sem_max["tasa_desercion"]:.1f}% deserción)')
print('→ La deserción es más frecuente en primeros semestres (adaptación universitaria).')
print('→ Los desertores tienen más atenciones psicológicas, indicando estrés previo.')

---
## Gráfica 9 (Bonus) — Perfil comparativo multi-variable

**Objetivo:** Mostrar un resumen visual del perfil promedio de un estudiante activo vs desertor en las variables clave.

**¿Por qué importa?** Presenta de forma clara y ejecutiva las diferencias clave para comunicar hallazgos al equipo.

In [ ]:
vars_clave = {
    'promedio_academico': 'Promedio\nacadémico',
    'asistencia_clases': 'Asistencia\n(%)',
    'materias_perdidas': 'Materias\nperdidas',
    'semestre_cursado': 'Semestre\ncursado',
    'atenciones_psicologicas': 'Atenciones\npsicológicas',
    'horas_trabajo_semanales': 'Horas trabajo\n/semana'
}

activos   = df[df['estado_estudiante'] == 'Activo']
desertores = df[df['estado_estudiante'] == 'Desertor']

medias_a = [activos[v].mean() for v in vars_clave]
medias_d = [desertores[v].mean() for v in vars_clave]
etiquetas = list(vars_clave.values())

x = np.arange(len(etiquetas))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 6))
bars_a = ax.bar(x - width/2, medias_a, width, label='Activo',   color='#2196F3', edgecolor='white')
bars_d = ax.bar(x + width/2, medias_d, width, label='Desertor', color='#F44336', edgecolor='white')

for bar in bars_a:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9, color='#1565C0')
for bar in bars_d:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=9, color='#B71C1C')

ax.set_xticks(x)
ax.set_xticklabels(etiquetas, fontsize=11)
ax.set_title('Gráfica 9 — Perfil comparativo: Activo vs Desertor', fontsize=14, fontweight='bold')
ax.set_ylabel('Valor promedio')
ax.legend(fontsize=11)

plt.tight_layout()
plt.savefig('grafica_09_perfil_comparativo.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📊 HALLAZGO — Perfil del estudiante en riesgo de deserción:')
for v, label in vars_clave.items():
    ma = activos[v].mean()
    md = desertores[v].mean()
    diff = ((md - ma) / ma * 100) if ma != 0 else 0
    print(f'  {label.replace(chr(10)," "):30s} Activo={ma:.1f} | Desertor={md:.1f} | Diferencia={diff:+.1f}%')

---
## Resumen de hallazgos y recomendaciones para el modelado

### Variables con mayor poder predictivo (por correlación y separación visual):

| Variable | Dirección | Importancia estimada |
|---|---|---|
| `promedio_academico` | Desertores tienen menor promedio | ⭐⭐⭐ Alta |
| `asistencia_clases` | Desertores tienen menor asistencia | ⭐⭐⭐ Alta |
| `materias_perdidas` | Desertores pierden más materias | ⭐⭐⭐ Alta |
| `mora_matricula` | Desertores tienen más mora | ⭐⭐ Media |
| `estrato` | Estratos bajos tienen mayor riesgo | ⭐⭐ Media |
| `atenciones_psicologicas` | Desertores tienen más atenciones | ⭐⭐ Media |
| `semestre_cursado` | Mayor riesgo en semestres iniciales | ⭐ Baja |

### Sobre el desbalanceo de clases:
- Dataset: **70% Activos / 30% Desertores** → desbalanceo moderado
- Recomendación: aplicar `class_weight='balanced'` en el modelo o SMOTE para oversampling
- Métrica recomendada: **F1-score** y **AUC-ROC** (no accuracy simple)

### Variables a excluir del modelo:
- `id_estudiante`, `primer_nombre`, `segundo_nombre`, etc. → identificadores, no predictores
- `historial_notas` → formato texto, requiere feature engineering adicional

### Nota técnica:
> Se identificó y corrigió un bug en `scripts/academics.py`: `np.arange(1, 20)` generaba 19 elementos incompatibles con el vector de probabilidades de 10 elementos. Corregido a `np.arange(1, 11)`. Esta corrección es parte del entregable de HU-01.